# 02 - Features e Split Temporal

Objetivo: entender como o projeto separa treino, validação e teste por tempo e como cria features sem vazamento de informação.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config.settings import Settings
from src.data.load_data import RawDataRepository
from src.data.merge_data import FraudDataMerger
from src.data.split_data import TemporalSplitter
from src.features.cleaning import FraudDataCleaner
from src.features.feature_engineering import FraudFeatureEngineer
from src.features.preprocessing import build_preprocessor, columns_to_drop

pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid")
settings = Settings(project_root=PROJECT_ROOT)

## 1. Preparar dataset consolidado

In [ ]:
raw = RawDataRepository(settings).load_all()
merged = FraudDataMerger(settings).merge(
    transactions=raw["transactions"],
    cards=raw["cards"],
    users=raw["users"],
    mcc_codes=raw["mcc"],
    labels=raw["labels"],
)
cleaned = FraudDataCleaner(settings).fit_transform(merged)
cleaned.shape

## 2. Split temporal

O teste fica no período mais recente. O threshold é escolhido na validação, nunca no teste.

In [ ]:
splits = TemporalSplitter(settings).split(cleaned)

split_summary = pd.DataFrame(
    [
        {"split": "train", "rows": len(splits.train), "start": splits.train[splits.time_column].min(), "end": splits.train[splits.time_column].max(), "fraud_rate": splits.train[settings.target_column].mean()},
        {"split": "validation", "rows": len(splits.validation), "start": splits.validation[splits.time_column].min(), "end": splits.validation[splits.time_column].max(), "fraud_rate": splits.validation[settings.target_column].mean()},
        {"split": "test", "rows": len(splits.test), "start": splits.test[splits.time_column].min(), "end": splits.test[splits.time_column].max(), "fraud_rate": splits.test[settings.target_column].mean()},
    ]
)
split_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=split_summary, x="split", y="fraud_rate", ax=ax)
ax.set_title("Taxa de fraude por partição temporal")
ax.set_ylabel("fraud_rate")
plt.show()

## 3. Engenharia de features sem dados futuros

O transformer calcula features comportamentais com `shift(1)`, isto é, usando transações anteriores ao registro atual.

In [ ]:
X_train = splits.train.drop(columns=[settings.target_column])
X_val = splits.validation.drop(columns=[settings.target_column])

feature_engineer = FraudFeatureEngineer(settings)
train_features = feature_engineer.fit_transform(X_train)
val_features = feature_engineer.transform(X_val)

created_columns = sorted(set(train_features.columns) - set(X_train.columns))
created_columns

In [ ]:
preview_columns = [col for col in ["amount", "previous_amount", "amount_mean_5_prev", "amount_std_5_prev", "amount_to_mean_5_prev", "transactions_seen_before"] if col in train_features.columns]
train_features[preview_columns].head(10)

## 4. Colunas removidas antes do modelo

IDs crus e colunas temporais brutas são removidos. O modelo usa features derivadas, numéricas e categóricas tratadas pelo `ColumnTransformer`.

In [ ]:
drop_cols = columns_to_drop(train_features.columns, settings)
drop_cols

In [ ]:
preprocessor = build_preprocessor(train_features, settings)
preprocessor

In [ ]:
numeric_cols = preprocessor.transformers[0][2]
categorical_cols = preprocessor.transformers[1][2]

pd.DataFrame(
    {
        "group": ["numeric", "categorical", "drop"],
        "count": [len(numeric_cols), len(categorical_cols), len(drop_cols)],
    }
)

## 5. Missing values nas features criadas

Nulos em features históricas são esperados para a primeira transação de um cartão/usuário. Eles são tratados por imputação dentro do preprocessador.

In [ ]:
feature_missing = (
    train_features[created_columns]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
feature_missing.columns = ["feature", "missing_rate"]
feature_missing